# Лабораторная работа № 3
## Тестирование чисел на простоту и построение больших простых чисел
### Вариант 5 — тест на основе малой теоремы Ферма

**Цель работы:** освоить основные программные методы тестирования чисел на простоту.

В работе реализуется генерация числа заданной битовой длины и его проверка на простоту. Для варианта 5 используется **тест на основе малой теоремы Ферма**.

Реализация следует заданию из методических указаний (стр. 7–8): случайное `p`-битовое число; установка старшего и младшего битов в `1`; проверка делимости на перечисленные в методичке малые простые числа, не превосходящие 256; пять проверок простоты. В общей инструкции методички назван тест Рабина—Миллера, но для варианта 5 он заменяется тестом Ферма согласно таблице вариантов на стр. 8.

## 1. Теоретическая основа теста Ферма

Малая теорема Ферма утверждает: если `n` — простое число, то для любого целого `a`, взаимно простого с `n`, выполняется

\[
a^{n-1} \equiv 1 \pmod n.
\]

На основе этого утверждения выполняется вероятностная проверка:

1. Случайно выбирается основание `a` из интервала \([2; n-1]\).
2. Вычисляется \(\operatorname{НОД}(a,n)\). Если НОД не равен `1`, число `n` составное.
3. Проверяется сравнение \(a^{n-1} \equiv 1 \pmod n\). Если оно не выполняется, число `n` составное.
4. Если сравнение выполняется, результат проверки — **неизвестно**. Проверка повторяется с другим случайным основанием.

Для варианта 5 выполняются пять проверок. Если хотя бы одна проверка обнаруживает составность, текущий кандидат отбрасывается и генерируется новый.

Обратное утверждение малой теоремы Ферма неверно: некоторые составные числа могут пройти проверку для всех взаимно простых с ними оснований (например, числа Кармайкла). Поэтому успешное прохождение пяти проверок означает, что число **вероятно простое**, но не является строгим доказательством простоты.

Для вычисления \(a^{n-1} \bmod n\) используется `pow(a, n - 1, n)`, которая вычисляет остаток от степени без построения полного большого значения степени.

## 2. Предварительная проверка делимости

Перед вероятностным тестом кандидат проверяется на делимость на **все простые числа меньше 2000**. Полный список заранее записан в массив `SMALL_PRIMES`; во время выполнения программы дополнительный алгоритм построения списка не используется. В массиве 303 простых числа.

Кандидат отбрасывается, если он делится хотя бы на одно из этих чисел и при этом не равен самому делителю.

In [7]:
SMALL_PRIMES = [
    2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37,
    41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89,
    97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151,
    157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223,
    227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281,
    283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359,
    367, 373, 379, 383, 389, 397, 401, 409, 419, 421, 431, 433,
    439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499, 503,
    509, 521, 523, 541, 547, 557, 563, 569, 571, 577, 587, 593,
    599, 601, 607, 613, 617, 619, 631, 641, 643, 647, 653, 659,
    661, 673, 677, 683, 691, 701, 709, 719, 727, 733, 739, 743,
    751, 757, 761, 769, 773, 787, 797, 809, 811, 821, 823, 827,
    829, 839, 853, 857, 859, 863, 877, 881, 883, 887, 907, 911,
    919, 929, 937, 941, 947, 953, 967, 971, 977, 983, 991, 997,
    1009, 1013, 1019, 1021, 1031, 1033, 1039, 1049, 1051, 1061, 1063, 1069,
    1087, 1091, 1093, 1097, 1103, 1109, 1117, 1123, 1129, 1151, 1153, 1163,
    1171, 1181, 1187, 1193, 1201, 1213, 1217, 1223, 1229, 1231, 1237, 1249,
    1259, 1277, 1279, 1283, 1289, 1291, 1297, 1301, 1303, 1307, 1319, 1321,
    1327, 1361, 1367, 1373, 1381, 1399, 1409, 1423, 1427, 1429, 1433, 1439,
    1447, 1451, 1453, 1459, 1471, 1481, 1483, 1487, 1489, 1493, 1499, 1511,
    1523, 1531, 1543, 1549, 1553, 1559, 1567, 1571, 1579, 1583, 1597, 1601,
    1607, 1609, 1613, 1619, 1621, 1627, 1637, 1657, 1663, 1667, 1669, 1693,
    1697, 1699, 1709, 1721, 1723, 1733, 1741, 1747, 1753, 1759, 1777, 1783,
    1787, 1789, 1801, 1811, 1823, 1831, 1847, 1861, 1867, 1871, 1873, 1877,
    1879, 1889, 1901, 1907, 1913, 1931, 1933, 1949, 1951, 1973, 1979, 1987,
    1993, 1997, 1999,
]


def passes_small_prime_check(n: int) -> bool:
    """Check divisibility by every prime number smaller than 2000.

    Args:
        n: Integer candidate.

    Returns:
        True if no prime in SMALL_PRIMES, other than n itself, divides n.
    """
    for prime in SMALL_PRIMES:
        if n != prime and n % prime == 0:
            return False
    return True


print(f"Количество простых чисел в массиве: {len(SMALL_PRIMES)}")
print(f"Проверяемый диапазон: от {SMALL_PRIMES[0]} до {SMALL_PRIMES[-1]}")
print("Массив простых чисел:", SMALL_PRIMES)

Количество простых чисел в массиве: 303
Проверяемый диапазон: от 2 до 1999
Массив простых чисел: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499, 503, 509, 521, 523, 541, 547, 557, 563, 569, 571, 577, 587, 593, 599, 601, 607, 613, 617, 619, 631, 641, 643, 647, 653, 659, 661, 673, 677, 683, 691, 701, 709, 719, 727, 733, 739, 743, 751, 757, 761, 769, 773, 787, 797, 809, 811, 821, 823, 827, 829, 839, 853, 857, 859, 863, 877, 881, 883, 887, 907, 911, 919, 929, 937, 941, 947, 953, 967, 971, 977, 983, 991, 997, 1009, 1013, 1019, 1021, 1031, 1033, 1039, 1049, 1051, 1061, 1063, 1069, 1087, 1091, 1093, 1

## 3. Один раунд теста Ферма

Для одного основания `a` последовательно проверяются НОД и сравнение по модулю. Основание выбирается случайно из закрытого интервала \([2; n-1]\), как указано в описании теста на стр. 2 методички.

Если НОД не равен единице или остаток от возведения в степень не равен единице, тест обнаружил составность.

In [8]:
from math import gcd
import secrets


def fermat_round(n: int, a: int) -> tuple[bool, int, int | None]:
    """Perform one test using the Fermat congruence.

    Args:
        n: Integer candidate greater than 2.
        a: Test base from the interval [2, n - 1].

    Returns:
        A tuple (passed, gcd_value, residue). A failed round means n is
        composite. A passed round means this base did not reveal compositeness.
    """
    if n <= 2:
        raise ValueError("Кандидат n должен быть больше 2.")
    if not 2 <= a <= n - 1:
        raise ValueError("Основание a должно находиться в интервале [2, n - 1].")

    gcd_value = gcd(a, n)
    if gcd_value != 1:
        return False, gcd_value, None

    residue = pow(a, n - 1, n)
    return residue == 1, gcd_value, residue


# Пример одного раунда на кандидате 101.
example_n = 101
example_a = 2
example_result = fermat_round(example_n, example_a)
print(f"n={example_n}, a={example_a}")
print(f"Раунд пройден: {example_result[0]}")
print(f"НОД(a, n): {example_result[1]}")
print(f"a^(n-1) mod n: {example_result[2]}")

n=101, a=2
Раунд пройден: True
НОД(a, n): 1
a^(n-1) mod n: 1


## 4. Пять проверок Ферма

Для одного кандидата случайно выбираются основания и выполняются пять раундов. Если раунд выявил составность, дальнейшие раунды для этого кандидата не нужны: по условию генерации следует перейти к новому `n`. Если все пять раундов пройдены, кандидат считается вероятно простым.

In [9]:
FERMAT_ROUNDS = 5


def fermat_primality_test(n: int) -> tuple[bool, list[dict]]:
    """Run five independent Fermat rounds for n.

    Args:
        n: Odd integer candidate greater than 2.

    Returns:
        A tuple (passed, details). passed is True only if all five rounds
        pass. details stores the base and intermediate values of each round.
    """
    details = []

    for round_number in range(1, FERMAT_ROUNDS + 1):
        # secrets.randbelow(n - 2) + 2 selects a uniformly random a in [2, n - 1].
        a = secrets.randbelow(n - 2) + 2
        passed, gcd_value, residue = fermat_round(n, a)

        details.append({
            "round": round_number,
            "a": a,
            "gcd": gcd_value,
            "residue": residue,
            "passed": passed,
        })

        if not passed:
            return False, details

    return True, details

## 5. Генерация случайного `p`-битового кандидата

Сначала генерируется случайное `p`-битовое число. Затем:
- старший бит устанавливается в `1`, чтобы число имело ровно `p` бит;
- младший бит устанавливается в `1`, чтобы число было нечётным.

Это соответствует первым двум шагам задания на стр. 7 методички.

In [10]:
def generate_p_bit_candidate(p: int) -> int:
    """Generate an odd random integer with exactly p bits.

    Args:
        p: Required bit length (p >= 2).

    Returns:
        A random p-bit odd integer.
    """
    if p < 2:
        raise ValueError("Битовая длина p должна быть не меньше 2.")

    n = secrets.randbits(p)
    n |= 1 << (p - 1)  # Установить старший бит в 1.
    n |= 1             # Установить младший бит в 1.
    return n


# Проверка формирования p-битового нечётного кандидата.
P = 32
candidate_example = generate_p_bit_candidate(P)
print("Кандидат:", candidate_example)
print("Битовая длина:", candidate_example.bit_length())
print("Нечётное число:", candidate_example % 2 == 1)

Кандидат: 2303465741
Битовая длина: 32
Нечётное число: True


## 6. Алгоритм генерации вероятно простого числа

Полный порядок действий:

1. Сгенерировать случайного нечётного кандидата заданной битовой длины `p`.
2. Последовательно проверить делимость кандидата на все 303 простых числа из массива `SMALL_PRIMES` (простые числа меньше 2000).
3. Выполнить тест Ферма пять раз, каждый раз выбирая новое случайное основание.
4. Если предварительная проверка или любой раунд Ферма выявили составность, сгенерировать новый кандидат и повторить процедуру.
5. Если кандидат прошёл все пять раундов, вернуть его.

При предварительной проверке перебираются все элементы массива `SMALL_PRIMES`; список не вычисляется программой во время выполнения.

In [11]:
def generate_probable_prime(p: int, max_attempts: int = 100_000) -> tuple[int, int, list[dict]]:
    """Generate a p-bit candidate that passes the small-prime and Fermat tests.

    Args:
        p: Required bit length.
        max_attempts: Maximum number of candidates to generate.

    Returns:
        A tuple (candidate, attempts, fermat_details).

    Raises:
        ValueError: If p or max_attempts is outside the supported range.
        RuntimeError: If no candidate is found within max_attempts.
    """
    if p < 2:
        raise ValueError("Битовая длина p должна быть не меньше 2.")
    if max_attempts < 1:
        raise ValueError("max_attempts должно быть положительным.")

    for attempt in range(1, max_attempts + 1):
        n = generate_p_bit_candidate(p)

        # Шаг 3 задания: предварительная проверка делимости на малые простые числа.
        if not passes_small_prime_check(n):
            continue

        # Для оставшихся кандидатов выполняются пять проверок Ферма.
        passed, details = fermat_primality_test(n)
        if passed:
            return n, attempt, details

    raise RuntimeError(
        f"Не удалось найти подходящий кандидат за {max_attempts} попыток."
    )

## 7. Запуск программы

Зададим битовую длину `p = 32` и запустим генерацию. Значение `P` можно изменить, чтобы получить число другой битовой длины.

In [12]:
P = 32

prime_candidate, attempts_count, fermat_details = generate_probable_prime(P)

print("Найденный кандидат:", prime_candidate)
print("Двоичная запись:", bin(prime_candidate))
print("Требуемая битовая длина:", P)
print("Фактическая битовая длина:", prime_candidate.bit_length())
print("Число нечётное:", prime_candidate % 2 == 1)
print("Количество сгенерированных кандидатов:", attempts_count)
print()
print("Пять раундов теста Ферма для найденного кандидата:")
for item in fermat_details:
    print(
        f"Раунд {item['round']}: a={item['a']}, НОД(a, n)={item['gcd']}, "
        f"a^(n-1) mod n={item['residue']}, результат={'пройден' if item['passed'] else 'не пройден'}"
    )

assert prime_candidate.bit_length() == P
assert prime_candidate % 2 == 1
assert passes_small_prime_check(prime_candidate)
assert len(fermat_details) == FERMAT_ROUNDS
assert all(item["passed"] for item in fermat_details)
print("\nКонтрольные условия выполнены.")

Найденный кандидат: 3229549159
Двоичная запись: 0b11000000011111110000001001100111
Требуемая битовая длина: 32
Фактическая битовая длина: 32
Число нечётное: True
Количество сгенерированных кандидатов: 1

Пять раундов теста Ферма для найденного кандидата:
Раунд 1: a=338746946, НОД(a, n)=1, a^(n-1) mod n=1, результат=пройден
Раунд 2: a=2614207298, НОД(a, n)=1, a^(n-1) mod n=1, результат=пройден
Раунд 3: a=3069939287, НОД(a, n)=1, a^(n-1) mod n=1, результат=пройден
Раунд 4: a=2279720513, НОД(a, n)=1, a^(n-1) mod n=1, результат=пройден
Раунд 5: a=494880142, НОД(a, n)=1, a^(n-1) mod n=1, результат=пройден

Контрольные условия выполнены.


## 8. Вывод

В ходе лабораторной работы реализована генерация числа заданной битовой длины и его вероятностная проверка на простоту для **варианта 5**.

Программа выполняет все этапы выбранного алгоритма:
- генерирует случайное `p`-битовое число;
- устанавливает старший и младший биты в `1`;
- проверяет делимость на все 303 простых числа меньше 2000, заранее записанные в массив `SMALL_PRIMES`;
- выполняет пять раундов теста на основе малой теоремы Ферма со случайными основаниями;
- при обнаружении составности отбрасывает кандидата и генерирует новый.

В результате получается нечётный кандидат требуемой битовой длины, который не делится ни на одно простое число из массива и прошёл пять раундов теста Ферма. Поэтому программа считает его **вероятно простым**. Тест Ферма не является строгим доказательством простоты: некоторые составные числа, в частности числа Кармайкла, могут проходить такие проверки.